# Modeling — Loan Default Prediction

This notebook trains and compares two gradient-boosted models (XGBoost and LightGBM) on the processed Home Credit dataset. Class imbalance is handled with SMOTE on the training set only, and the two models are compared on AUC-ROC, classification report, ROC curves, precision-recall curves, and feature importance. Trained models are saved to `../models/` for use in the explainability dashboard.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    classification_report,
)
from imblearn.over_sampling import SMOTE

import xgboost as xgb
import lightgbm as lgb

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

MODELS_DIR = "../models"
os.makedirs(MODELS_DIR, exist_ok=True)

RANDOM_STATE = 42


## 2. Load Processed Data

In [2]:
df = pd.read_csv("../data/processed/train_processed.csv")
print(f"Loaded shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
df.head()


Loaded shape: 307,511 rows x 190 columns


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,ORGANIZATION_TYPE_Trade: type 4,ORGANIZATION_TYPE_Trade: type 5,ORGANIZATION_TYPE_Trade: type 6,ORGANIZATION_TYPE_Trade: type 7,ORGANIZATION_TYPE_Transport: type 1,ORGANIZATION_TYPE_Transport: type 2,ORGANIZATION_TYPE_Transport: type 3,ORGANIZATION_TYPE_Transport: type 4,ORGANIZATION_TYPE_University,ORGANIZATION_TYPE_XNA
0,100002,1,0,0,0,0,202500.0,406597.5,24700.5,351000.0,...,0,0,0,0,0,0,0,0,0,0
1,100003,0,0,0,1,0,270000.0,1293502.5,35698.5,1129500.0,...,0,0,0,0,0,0,0,0,0,0
2,100004,0,1,1,0,0,67500.0,135000.0,6750.0,135000.0,...,0,0,0,0,0,0,0,0,0,0
3,100006,0,0,0,0,0,135000.0,312682.5,29686.5,297000.0,...,0,0,0,0,0,0,0,0,0,0
4,100007,0,0,0,0,0,121500.0,513000.0,21865.5,513000.0,...,0,0,0,0,0,0,0,0,0,0


## 3. Split into X and y

In [3]:
X = df.drop(columns=["TARGET"])
y = df["TARGET"]

# Fix special characters in column names for LightGBM
X.columns = X.columns.str.replace(r'[^A-Za-z0-9_]', '_', regex=True)

print(f"X shape: {X.shape}")
print(f"y distribution:\n{y.value_counts()}")

X shape: (307511, 189)
y distribution:
TARGET
0    282686
1     24825
Name: count, dtype: int64


## 4. Train/Test Split (80/20, Stratified)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train target rate: {y_train.mean() * 100:.2f}%")
print(f"Test target rate: {y_test.mean() * 100:.2f}%")


Train shape: (246008, 189), Test shape: (61503, 189)
Train target rate: 8.07%
Test target rate: 8.07%


## 5. Handle Class Imbalance — SMOTE (Training Data Only)

SMOTE is fit and applied only to the training split so that the test set continues to reflect the real-world class distribution.

In [5]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE: {X_train.shape}, target distribution:\n{y_train.value_counts()}")
print(f"\nAfter SMOTE: {X_train_res.shape}, target distribution:\n{y_train_res.value_counts()}")


Before SMOTE: (246008, 189), target distribution:
TARGET
0    226148
1     19860
Name: count, dtype: int64

After SMOTE: (452296, 189), target distribution:
TARGET
0    226148
1    226148
Name: count, dtype: int64


## 6. Train XGBoost Model

In [ ]:
scale_pos_weight = (y_train_res.value_counts()[0] / y_train_res.value_counts()[1])
print(f"scale_pos_weight: {scale_pos_weight:.3f}")

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="auc",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

xgb_model.fit(X_train_res, y_train_res)
print("XGBoost training complete.")


scale_pos_weight: 1.000


## 7. Train LightGBM Model

In [ ]:
lgbm_model = lgb.LGBMClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

lgbm_model.fit(X_train_res, y_train_res)
print("LightGBM training complete.")


## 8. Evaluate Models — AUC-ROC and Classification Report

In [ ]:
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
xgb_pred = xgb_model.predict(X_test)
xgb_auc = roc_auc_score(y_test, xgb_proba)

lgbm_proba = lgbm_model.predict_proba(X_test)[:, 1]
lgbm_pred = lgbm_model.predict(X_test)
lgbm_auc = roc_auc_score(y_test, lgbm_proba)

print("=" * 60)
print(f"XGBoost  — AUC-ROC: {xgb_auc:.4f}")
print("=" * 60)
print(classification_report(y_test, xgb_pred, target_names=["No Default", "Default"]))

print("=" * 60)
print(f"LightGBM — AUC-ROC: {lgbm_auc:.4f}")
print("=" * 60)
print(classification_report(y_test, lgbm_pred, target_names=["No Default", "Default"]))


## 9. ROC Curves

In [ ]:
xgb_fpr, xgb_tpr, _ = roc_curve(y_test, xgb_proba)
lgbm_fpr, lgbm_tpr, _ = roc_curve(y_test, lgbm_proba)

fig, ax = plt.subplots(figsize=(8, 7))
ax.plot(xgb_fpr, xgb_tpr, label=f"XGBoost (AUC = {xgb_auc:.4f})", color="#4C72B0", linewidth=2)
ax.plot(lgbm_fpr, lgbm_tpr, label=f"LightGBM (AUC = {lgbm_auc:.4f})", color="#C44E52", linewidth=2)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random baseline")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve — XGBoost vs LightGBM")
ax.legend(loc="lower right")

plt.tight_layout()
plt.savefig("../data/processed/roc_curves.png", bbox_inches="tight")
plt.show()


## 10. Precision-Recall Curves

In [ ]:
xgb_prec, xgb_rec, _ = precision_recall_curve(y_test, xgb_proba)
lgbm_prec, lgbm_rec, _ = precision_recall_curve(y_test, lgbm_proba)

fig, ax = plt.subplots(figsize=(8, 7))
ax.plot(xgb_rec, xgb_prec, label="XGBoost", color="#4C72B0", linewidth=2)
ax.plot(lgbm_rec, lgbm_prec, label="LightGBM", color="#C44E52", linewidth=2)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve — XGBoost vs LightGBM")
ax.legend(loc="upper right")

plt.tight_layout()
plt.savefig("../data/processed/precision_recall_curves.png", bbox_inches="tight")
plt.show()


## 11. Feature Importance (Top 20)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

xgb_importance = pd.Series(xgb_model.feature_importances_, index=X.columns)
xgb_importance = xgb_importance.sort_values(ascending=False).head(20).sort_values()
axes[0].barh(xgb_importance.index, xgb_importance.values, color="#4C72B0")
axes[0].set_title("XGBoost — Top 20 Feature Importances")
axes[0].set_xlabel("Importance")

lgbm_importance = pd.Series(lgbm_model.feature_importances_, index=X.columns)
lgbm_importance = lgbm_importance.sort_values(ascending=False).head(20).sort_values()
axes[1].barh(lgbm_importance.index, lgbm_importance.values, color="#C44E52")
axes[1].set_title("LightGBM — Top 20 Feature Importances")
axes[1].set_xlabel("Importance")

plt.tight_layout()
plt.savefig("../data/processed/feature_importance.png", bbox_inches="tight")
plt.show()


## 12. Save Trained Models

In [ ]:
xgb_path = f"{MODELS_DIR}/xgb_model.pkl"
lgbm_path = f"{MODELS_DIR}/lgbm_model.pkl"

joblib.dump(xgb_model, xgb_path)
joblib.dump(lgbm_model, lgbm_path)

print(f"Saved XGBoost model to {xgb_path}")
print(f"Saved LightGBM model to {lgbm_path}")


## 13. Final Comparison Summary

In [ ]:
print("=" * 60)
print("MODEL COMPARISON SUMMARY")
print("=" * 60)

summary = pd.DataFrame({
    "Model": ["XGBoost", "LightGBM"],
    "AUC-ROC": [xgb_auc, lgbm_auc],
})
print(summary.to_string(index=False))

best_model = "XGBoost" if xgb_auc >= lgbm_auc else "LightGBM"
print(f"\nBest model by AUC-ROC: {best_model}")
print(f"Models saved to: {MODELS_DIR}/")
